# Python Excel Automation: การใช้งาน Workbook และ Worksheet ใน OpenPyXl - ตอนที่ 2

**คำอธิบาย:** เรียนรู้การใช้งาน Workbook และ Worksheet ผ่านตัวอย่าง Python ต่างๆ

**หัวข้อที่ครอบคลุม:**
- ทำความเข้าใจโหมดการเปิดไฟล์ Workbook (read_only, data_only)
- อ่านข้อมูลจาก Worksheet ด้วยการวนลูป
- สร้าง Worksheet ในตำแหน่งที่ต้องการ
- บันทึกและปิด Workbook
- วนลูปและเปลี่ยนชื่อ Worksheet
- คัดลอกและย้าย Worksheet
- ลบ Worksheet
- เลือก Worksheet ด้วย Index

**หมายเหตุ:** openpyxl ไม่รองรับไฟล์ .xls หรือ .csv รองรับเฉพาะ .xlsx เท่านั้น

In [ ]:
# 📦 ติดตั้ง packages ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import openpyxl
    print("✅ ติดตั้ง packages ทั้งหมดแล้ว")
except ImportError:
    %pip install openpyxl

In [ ]:
# 📚 นำเข้า openpyxl โดยตั้งชื่อย่อเป็น 'excel' เพื่อความสะดวก
# openpyxl ไม่ต้องติดตั้ง Excel — เป็น Python ล้วน
# ⚠️ รองรับเฉพาะไฟล์ .xlsx (ไม่รองรับ .xls หรือ .csv)
import openpyxl as excel

## 1. โหมดการเปิดไฟล์ Workbook (File Modes)

In [ ]:
# 📝 สร้าง workbook ตัวอย่างสำหรับสาธิต
# .append() เพิ่มแถวใหม่ที่ด้านล่าง — เหมาะสำหรับสร้างข้อมูลทีละแถว
# เราเพิ่มสูตร SUM เพื่อสาธิต data_only mode ในภายหลัง
wb_demo = openpyxl.Workbook()
ws = wb_demo.active
ws.title = 'Orders'
ws.append(['Order ID', 'Product', 'Amount', 'State'])
ws.append([1001, 'Widget A', 150.50, 'California'])
ws.append([1002, 'Widget B', 230.00, 'Texas'])
ws.append([1003, 'Widget C', 89.99, 'New York'])
ws.append([1004, 'Widget A', 150.50, 'California'])
ws.append([1005, 'Widget D', 475.00, 'Florida'])

# เพิ่มสูตร (openpyxl เก็บสูตรเป็น string)
ws['E1'] = 'Total'
ws['E2'] = '=SUM(C2:C6)'

wb_demo.save('superstore.xlsx')
print("✅ สร้าง workbook ตัวอย่าง: superstore.xlsx")

In [ ]:
%%time
# 📖 โหมดที่ 1: READ-ONLY — เปิดไฟล์ใหญ่ได้เร็วกว่า แต่แก้ไขไม่ได้
# read_only=True ข้ามการโหลดสไตล์ สูตร ฯลฯ — อ่านแค่ข้อมูล
# ต้อง .close() เมื่อใช้เสร็จเพื่อปล่อยไฟล์
# ซ่อน Warning จาก openpyxl เรื่อง extensions ที่ไม่รองรับ
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    wkb_readonly = excel.load_workbook('superstore.xlsx', read_only=True)
print(f"โหมด read-only: {wkb_readonly.read_only}")
print(f"Sheet ทั้งหมด: {wkb_readonly.sheetnames}")
wkb_readonly.close()

In [ ]:
# 📊 โหมดที่ 2: DATA-ONLY — อ่านค่าที่คำนวณแล้ว แทนที่จะเป็นสูตร
# data_only=True: ส่งกลับผลลัพธ์ของสูตร (แสดง None ถ้าไม่เคยเปิดใน Excel)
# ไม่ใช้ data_only: ส่งกลับสูตรเป็น string เช่น "=SUM(C2:C6)"
wkb_data = excel.load_workbook('superstore.xlsx', data_only=True)
ws_data = wkb_data['Orders']
print(f"E2 ด้วย data_only=True: {ws_data['E2'].value}")
wkb_data.close()

# โหมดปกติ — อ่านข้อความสูตรจริง
wkb_formula = excel.load_workbook('superstore.xlsx')
ws_formula = wkb_formula['Orders']
print(f"E2 แบบสูตร: {ws_formula['E2'].value}")
wkb_formula.close()

## 2. อ่านข้อมูลจาก Worksheet

In [ ]:
# 📋 โหลด workbook แล้วแสดงข้อมูลทั้งหมดด้วย iter_rows
# values_only=True ส่งกลับแค่ค่า (ไม่ใช่ cell objects)
wkb = excel.load_workbook('superstore.xlsx')
ws = wkb['Orders']

for row in ws.iter_rows(values_only=True):
    print(row)

In [ ]:
# 📐 ขนาด Worksheet และการเข้าถึงแถวเฉพาะ
# .dimensions = ช่วงที่ใช้งาน เช่น "A1:E6"
# ws[1] = แถวที่ 1 — ส่งกลับ Cell objects พร้อม .column_letter และ .value
print(f"ขนาดข้อมูล: {ws.dimensions}")
print(f"จำนวนแถว: {ws.max_row}")
print(f"จำนวนคอลัมน์: {ws.max_column}")

# ดึงแถวหัวตาราง (แถวที่ 1)
for cell in ws[1]:
    print(f"{cell.column_letter}: {cell.value}")

## 3. สร้าง Worksheet ในตำแหน่งที่ต้องการ

In [ ]:
# ➕ สร้าง Worksheet ในตำแหน่งที่กำหนด
# ตัวเลขตัวที่ 2 = index (เริ่มจาก 0): 0 = แรกสุด, 1 = ที่สอง ฯลฯ
# ถ้าไม่ระบุ index sheet ใหม่จะไปต่อท้าย
ws_first = wkb.create_sheet('First Sheet', 0)  # ตำแหน่ง 0 = แรกสุด
ws_last = wkb.create_sheet('Last Sheet')        # ค่าเริ่มต้น = ท้ายสุด
ws_middle = wkb.create_sheet('Middle Sheet', 1) # ตำแหน่ง 1 = ที่สอง

print(f"ลำดับ Sheet: {wkb.sheetnames}")

## 4. วนลูปและเปลี่ยนชื่อ Worksheet

In [ ]:
# 🔄 วนลูปผ่าน Worksheet ทั้งหมดด้วย wkb.worksheets
# enumerate() ให้ทั้ง index และ sheet object
for i, sheet in enumerate(wkb.worksheets):
    print(f"Index {i}: {sheet.title}")

In [ ]:
# ✏️ เปลี่ยนชื่อ Worksheet — แค่ตั้งค่า .title
wkb['First Sheet'].title = 'Renamed_First'
print(f"Sheet หลังเปลี่ยนชื่อ: {wkb.sheetnames}")

In [ ]:
# 🏷️ เปลี่ยนชื่อหลายตัว — เพิ่ม prefix ให้ทุก sheet (comment ไว้เพื่อป้องกันปัญหา)
# เอา comment ออกเพื่อลองใช้งาน
# for sheet in wkb.worksheets:
#     sheet.title = f"Report_{sheet.title}"
# print(f"Sheet ทั้งหมด: {wkb.sheetnames}")

## 5. คัดลอก Worksheet

In [ ]:
# 📋 คัดลอก Worksheet — สร้างสำเนาพร้อมข้อมูลและการจัดรูปแบบ
# .copy_worksheet() สร้างสำเนาชื่อเช่น "Orders Copy"
# ⚠️ ใช้ได้แค่ภายใน workbook เดียวกัน (คัดลอกข้าม workbook ไม่ได้)
source_ws = wkb['Orders']
copied_ws = wkb.copy_worksheet(source_ws)
print(f"ชื่อ sheet ที่คัดลอก: {copied_ws.title}")
print(f"Sheet ทั้งหมด: {wkb.sheetnames}")

## 6. ย้าย Worksheet

In [ ]:
# 🔀 ย้าย Worksheet — เลื่อนตำแหน่งในลำดับแท็บ
# offset: บวก = เลื่อนไปขวา, ลบ = เลื่อนไปซ้าย
# ตัวอย่าง: ย้าย "Orders" ไปทางซ้าย 1 ตำแหน่ง
wkb.move_sheet('Orders', offset=-1)
print(f"Sheet หลังย้าย: {wkb.sheetnames}")

## 7. ลบ Worksheet

In [ ]:
# 🗑️ ลบ Worksheet — ใช้ del wkb['ชื่อ sheet']
# ⚠️ ลบถาวร! ตรวจสอบก่อนเสมอว่า sheet มีอยู่จริงเพื่อป้องกัน KeyError
if 'Middle Sheet' in wkb.sheetnames:
    del wkb['Middle Sheet']
    print("ลบ 'Middle Sheet' แล้ว")

print(f"Sheet ที่เหลือ: {wkb.sheetnames}")

## 8. เลือก Worksheet ด้วย Index

In [ ]:
# 🎯 เลือก Worksheet ที่ใช้งาน — กำหนดว่า sheet ไหนเปิดก่อนเมื่อเปิดไฟล์
# wkb.active = index (0 = sheet แรก)
# หรือใช้ .sheetnames.index('ชื่อ') เพื่อหา index จากชื่อ
wkb.active = 0  # เลือก sheet แรก
print(f"Sheet ที่ใช้งาน: {wkb.active.title}")

wkb.active = wkb.sheetnames.index('Orders')
print(f"Sheet ที่ใช้งาน: {wkb.active.title}")

## 9. บันทึกและปิด

In [ ]:
# 💾 บันทึกและปิด workbook
# .save() เขียนการเปลี่ยนแปลงทั้งหมดลงดิสก์
# .close() ปล่อยไฟล์จากหน่วยความจำ
wkb.save('openpyxl_workbooks_worksheets.xlsx')
print("✅ บันทึกเป็น 'openpyxl_workbooks_worksheets.xlsx' แล้ว")

wkb.close()
print("✅ ปิด Workbook แล้ว")

---

## 🎮 Mini Project: จัดการไฟล์รายงานประจำเดือน

สมมติว่าคุณเป็น Admin ที่ต้องจัดการ Excel กับ Sheet ต่างๆ

> 💡 **คำตอบ:** ดูไฟล์ `answers/03_answers.ipynb`

---

### โจทย์ที่ 1: สร้าง Workbook รายงาน 12 เดือน 🟢
1. สร้าง Workbook ใหม่
2. ลบ sheet default (Sheet)
3. สร้าง sheet 12 อัน ชื่อ: `ม.ค.`, `ก.พ.`, `มี.ค.`, ... , `ธ.ค.`
4. ใน sheet แต่ละเดือน ใส่ header: `วันที่`, `รายการ`, `จำนวนเงิน`
5. บันทึกเป็น `monthly_report.xlsx`

> 💡 Hint: ใช้ list ชื่อเดือน + `for loop` + `create_sheet()`

In [ ]:
# โจทย์ที่ 1: สร้าง Workbook รายงาน 12 เดือน
# เขียนโค้ดของคุณที่นี่ 👇
from openpyxl import Workbook
import random

wb = Workbook()
# ลบ sheet default
del wb['Sheet']

months = ['ม.ค.', 'ก.พ.', 'มี.ค.', 'เม.ย.', 'พ.ค.', 'มิ.ย.',
          'ก.ค.', 'ส.ค.', 'ก.ย.', 'ต.ค.', 'พ.ย.', 'ธ.ค.']

# ตัวอย่างรายการ
items = [
    'ค่าอาหาร', 'ค่าเดินทาง', 'ค่าน้ำมัน', 
    'ค่าของใช้', 'ค่ากาแฟ', 'ค่าอินเทอร์เน็ต'
]

for i, month in enumerate(months, start=1):
    ws = wb.create_sheet(month)

    # header
    ws.append(['วันที่', 'รายการ', 'จำนวนเงิน'])

    # เพิ่มข้อมูลตัวอย่าง 5 แถว
    for day in range(1, 6):
        date = f'{day:02d}/{months.index(month)+1:02d}/2026'
        item = random.choice(items)
        amount = random.randint(50, 500)

        ws.append([date, item, amount])

wb.save('monthly_report.xlsx')
print(f'✅ สร้าง {len(wb.sheetnames)} sheet สำเร็จ!')
print(f'📋 Sheets: {wb.sheetnames}')

### โจทย์ที่ 2: จัดระเบียบ Sheet 🟡
เปิดไฟล์ `monthly_report.xlsx` แล้ว:
1. Copy sheet `ม.ค.` เป็น `ม.ค. (สำรอง)`
2. Move sheet `ธ.ค.` ไปอยู่ตำแหน่งแรกสุด
3. เปลี่ยนชื่อ sheet `ม.ค.` เป็น `January`
4. ตั้ง active sheet เป็น `ก.พ.`
5. Print รายชื่อ sheet ทั้งหมด
6. บันทึกไฟล์

> 💡 Hint: ใช้ `copy_worksheet()`, `move_sheet()`, `sheet.title =`, `wkb.active =`

In [ ]:
from openpyxl import load_workbook

wkb = load_workbook('monthly_report.xlsx')

# 1. Copy sheet ม.ค.
source = wkb['ม.ค.']
wkb.copy_worksheet(source)
wkb.worksheets[-1].title = 'ม.ค. (สำรอง)'

# 2. Move ธ.ค. ไปตำแหน่งแรก
dec_idx = wkb.sheetnames.index('ธ.ค.')
wkb.move_sheet('ธ.ค.', offset=-dec_idx)

# 3. เปลี่ยนชื่อ ม.ค. เป็น January
wkb['ม.ค.'].title = 'January'

# 4. ตั้ง active sheet เป็น ก.พ.
wkb.active = wkb.sheetnames.index('ก.พ.')

# 5. Print รายชื่อ
print(f'📋 Sheets: {wkb.sheetnames}')

wkb.save('monthly_report.xlsx')
print('✅ จัดระเบียบ sheet สำเร็จ!')

### โจทย์ที่ 3: ลบ Sheet ที่ไม่ต้องการ 🔴
เปิดไฟล์ `monthly_report.xlsx` แล้ว:
1. ลบทุก sheet ที่มีคำว่า `"สำรอง"` ในชื่อ
2. เก็บเฉพาะ sheet `January`, `ก.พ.`, `มี.ค.` (ลบเดือนที่เหลือออก)
3. Print จำนวน sheet ที่เหลือ
4. แสดง `dimensions` ของแต่ละ sheet
5. บันทึกเป็นไฟล์ใหม่ `q1_report.xlsx`

> 💡 Hint: ใช้ loop + `del wkb[sheet_name]` + เช็ค `if "สำรอง" in name`

In [ ]:
a = 1
print(a)

a = 10
print(a)

a = 50 
print(a)

In [ ]:
# โจทย์ที่ 3: ลบ Sheet ที่ไม่ต้องการ
# เขียนโค้ดของคุณที่นี่ 👇
from openpyxl import load_workbook

wkb = load_workbook('monthly_report.xlsx')

# 1. ลบ sheet ที่มีคำว่า สำรอง
to_delete = [name for name in wkb.sheetnames if 'สำรอง' in name]
for name in to_delete:
    del wkb[name]
    print(f'  🗑️ ลบ: {name}')

# 2. เก็บเฉพาะ January, ก.พ., มี.ค.
keep = ['January', 'ก.พ.', 'มี.ค.']
to_delete = [name for name in wkb.sheetnames if name not in keep]
for name in to_delete:
    del wkb[name]
    print(f'  🗑️ ลบ: {name}')

# 3. Print จำนวน sheet ที่เหลือ
print(f'\n📋 เหลือ {len(wkb.sheetnames)} sheets: {wkb.sheetnames}')

# 4. แสดง dimensions
for sheet in wkb.worksheets:
    print(f'  {sheet.title}: dimensions={sheet.dimensions}')

wkb.save('q1_report.xlsx')
print('\n✅ บันทึก q1_report.xlsx สำเร็จ!')